# LAG and LEAD

## Overview
`LAG` and `LEAD` are offset window functions that access a value from another row within the same partition, relative to the current row — without a self join.

```sql
LAG (expression [, offset [, default]])  OVER (PARTITION BY ... ORDER BY ...)
LEAD(expression [, offset [, default]])  OVER (PARTITION BY ... ORDER BY ...)
```

| Parameter | Default | Meaning |
|---|---|---|
| `expression` | — | Column or expression to retrieve |
| `offset` | 1 | How many rows back (LAG) or forward (LEAD) |
| `default` | NULL | Value returned when the offset row doesn't exist |

**LAG** looks backwards — gets the value from a previous row.
**LEAD** looks forwards — gets the value from a following row.

**Key use cases:**
- Period-over-period change (month-over-month, day-over-day)
- Time between events (days since last transaction)
- Detecting direction changes (price going up → down)
- Sessionisation (time gap between events)

**Both functions require `ORDER BY` inside `OVER`.** Without it, "previous" and "next" have no meaning.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT);
CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, province TEXT, opened_date TEXT, status TEXT);
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, account_id INTEGER, trade_date TEXT, ticker TEXT, direction TEXT, shares INTEGER, price REAL, total_value REAL);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),(102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),(104,3,'Investment','ON','2018-05-20','Active'),
  (105,4,'Chequing','NB','2015-09-30','Active'),(106,5,'Chequing','BC','2021-06-15','Active'),
  (107,6,'Chequing','AB','2017-11-22','Active'),(108,7,'Investment','ON','2016-03-08','Active'),
  (109,8,'Chequing','QC','2023-01-05','Active'),(110,9,'Investment','BC','2022-08-19','Active'),
  (111,10,'Chequing','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,101,'2023-03-05','Deposit',4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',15000.00,'Payroll',0),(1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',15000.00,'Payroll',0),(1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',15000.00,'Payroll',0),(1012,105,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1014,105,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),(1016,107,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',-25.00,'Fee',1),(1018,107,'2023-03-15','Withdrawal',-450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',2800.00,'Payroll',0),(1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',4500.00,'Payroll',0),(1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',4500.00,'Payroll',0);
INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy',10,148.50,1485.00),(2002,104,'2023-01-25','MSFT','Buy',5,240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy',5,153.20,766.00),(2004,104,'2023-02-28','MSFT','Sell',3,252.80,758.40),
  (2005,104,'2023-03-15','NVDA','Buy',2,278.50,557.00),(2006,108,'2023-01-05','AAPL','Buy',20,130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy',8,95.20,761.60),(2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy',5,99.80,499.00),(2010,108,'2023-03-10','NVDA','Buy',4,265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy',8,238.00,1904.00),(2012,110,'2023-02-01','AAPL','Buy',15,145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy',3,248.50,745.50),(2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy',6,280.00,1680.00);
""")
conn.commit()
print('Finance schema ready.')


Finance schema ready.


---
## Basic LAG — previous row value

In [2]:
# Previous transaction amount per account, and change since last transaction
print('=== Each transaction vs the previous one for the same account ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        LAG(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
        )                                                   AS prev_amount,
        ROUND(amount - LAG(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
        ), 2)                                               AS amount_change,
        LAG(txn_date) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
        )                                                   AS prev_txn_date,
        CAST(JULIANDAY(txn_date) - JULIANDAY(
            LAG(txn_date) OVER (
                PARTITION BY account_id
                ORDER BY txn_date, txn_id
            )
        ) AS INTEGER)                                       AS days_since_prev
FROM    transactions
WHERE   account_id IN (101, 103)
ORDER BY account_id, txn_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('First row per account has NULL prev_amount — no prior row exists in the partition.')


=== Each transaction vs the previous one for the same account ===
 txn_id  account_id   txn_date  amount  prev_amount  amount_change prev_txn_date  days_since_prev
   1001         101 2023-01-05  4200.0          NaN            NaN          None              NaN
   1002         101 2023-01-12  -850.0       4200.0        -5050.0    2023-01-05              7.0
   1003         101 2023-01-20  -120.0       -850.0          730.0    2023-01-12              8.0
   1004         101 2023-02-05  4200.0       -120.0         4320.0    2023-01-20             16.0
   1005         101 2023-02-18  -980.0       4200.0        -5180.0    2023-02-05             13.0
   1006         101 2023-03-05  4200.0       -980.0         5180.0    2023-02-18             15.0
   1007         103 2023-01-08 15000.0          NaN            NaN          None              NaN
   1008         103 2023-01-25 -3200.0      15000.0       -18200.0    2023-01-08             17.0
   1009         103 2023-02-08 15000.0      -3200.0 

---
## LAG default value — replacing the first-row NULL

In [3]:
# Use a default value instead of NULL for the first row
print('=== LAG with default value (0) for the first transaction ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        LAG(amount, 1, 0.0) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
        )                           AS prev_amount_or_zero,
        ROUND(amount - LAG(amount, 1, 0.0) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
        ), 2)                       AS change_from_prev
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Default 0.0 means the first transaction shows a change equal to its own amount.')
print('Whether this is appropriate depends on the business context.')


=== LAG with default value (0) for the first transaction ===
 txn_id  account_id   txn_date  amount  prev_amount_or_zero  change_from_prev
   1001         101 2023-01-05  4200.0                  0.0            4200.0
   1002         101 2023-01-12  -850.0               4200.0           -5050.0
   1003         101 2023-01-20  -120.0               -850.0             730.0
   1004         101 2023-02-05  4200.0               -120.0            4320.0
   1005         101 2023-02-18  -980.0               4200.0           -5180.0
   1006         101 2023-03-05  4200.0               -980.0            5180.0

Default 0.0 means the first transaction shows a change equal to its own amount.
Whether this is appropriate depends on the business context.


---
## LEAD — looking forward

In [4]:
# For each trade, show what the next trade price was for the same ticker
print('=== Each AAPL trade with the next AAPL trade price (LEAD) ===')
q = """
SELECT  trade_id,
        account_id,
        trade_date,
        ticker,
        direction,
        price,
        LEAD(price) OVER (
            PARTITION BY ticker
            ORDER BY trade_date, trade_id
        )                                                    AS next_price,
        ROUND(LEAD(price) OVER (
            PARTITION BY ticker
            ORDER BY trade_date, trade_id
        ) - price, 2)                                        AS price_chg_to_next,
        LEAD(trade_date) OVER (
            PARTITION BY ticker
            ORDER BY trade_date, trade_id
        )                                                    AS next_trade_date
FROM    trades
WHERE   ticker = 'AAPL'
ORDER BY trade_date, trade_id
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== Each AAPL trade with the next AAPL trade price (LEAD) ===
 trade_id  account_id trade_date ticker direction  price  next_price  price_chg_to_next next_trade_date
     2006         108 2023-01-05   AAPL       Buy  130.5       148.5               18.0      2023-01-10
     2001         104 2023-01-10   AAPL       Buy  148.5       145.0               -3.5      2023-02-01
     2012         110 2023-02-01   AAPL       Buy  145.0       151.0                6.0      2023-02-08
     2008         108 2023-02-08   AAPL      Sell  151.0       153.2                2.2      2023-02-14
     2003         104 2023-02-14   AAPL       Buy  153.2       155.0                1.8      2023-03-05
     2014         110 2023-03-05   AAPL      Sell  155.0         NaN                NaN            None


---
## Month-over-month change — a classic business report

In [5]:
# Monthly deposit totals with MoM change and % change
print('=== Month-over-month deposit total change ===')
q = """
WITH monthly AS (
    SELECT
        STRFTIME('%Y-%m', txn_date)  AS month,
        ROUND(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 2) AS total_deposits,
        COUNT(CASE WHEN amount > 0 THEN 1 END)                     AS deposit_count
    FROM transactions
    GROUP BY month
)
SELECT
    month,
    total_deposits,
    deposit_count,
    LAG(total_deposits) OVER (ORDER BY month)       AS prev_month_deposits,
    ROUND(total_deposits -
          LAG(total_deposits) OVER (ORDER BY month), 2) AS mom_change,
    ROUND(100.0 * (total_deposits -
          LAG(total_deposits) OVER (ORDER BY month))
          / NULLIF(LAG(total_deposits) OVER (ORDER BY month), 0), 1) AS mom_pct_change
FROM monthly
ORDER BY month
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Multi-step lag: compare to 2 periods ago
print()
print('=== LAG with offset=2: compare to 2 months prior ===')
q2 = """
WITH monthly AS (
    SELECT STRFTIME('%Y-%m', txn_date) AS month,
           ROUND(SUM(CASE WHEN amount>0 THEN amount ELSE 0 END),2) AS deposits
    FROM transactions GROUP BY month
)
SELECT  month, deposits,
        LAG(deposits, 1) OVER (ORDER BY month) AS prev_1_month,
        LAG(deposits, 2) OVER (ORDER BY month) AS prev_2_months
FROM monthly ORDER BY month
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


=== Month-over-month deposit total change ===
  month  total_deposits  deposit_count  prev_month_deposits  mom_change  mom_pct_change
2023-01         33400.0              6                  NaN         NaN             NaN
2023-02         22300.0              3              33400.0    -11100.0           -33.2
2023-03         23700.0              3              22300.0      1400.0             6.3

=== LAG with offset=2: compare to 2 months prior ===
  month  deposits  prev_1_month  prev_2_months
2023-01   33400.0           NaN            NaN
2023-02   22300.0       33400.0            NaN
2023-03   23700.0       22300.0        33400.0


---
## Common Pitfalls

**1. LAG/LEAD without ORDER BY is undefined**
`LAG(amount) OVER (PARTITION BY account_id)` without ORDER BY has no defined row ordering — "previous" is meaningless. SQLite may return an arbitrary value or error. Always include ORDER BY in the window specification when using LAG or LEAD.

**2. The first (LAG) and last (LEAD) rows return NULL by default**
The first row has no previous row for LAG, the last row has no following row for LEAD — both return NULL. Use the third parameter for a default: `LAG(amount, 1, 0)`. However, only use a non-NULL default when it makes business sense (0 may distort a rate calculation).

**3. Ties in ORDER BY make the offset non-deterministic**
If two transactions share the same date, `LAG(amount) OVER (ORDER BY txn_date)` could retrieve either one as the "previous" row. Add a unique tiebreaker (`txn_id`) to guarantee consistent results: `ORDER BY txn_date, txn_id`.

**4. Dividing by LAG value without NULLIF causes division errors on first rows**
`amount / LAG(amount) OVER (...)` produces a division-by-NULL (returns NULL) for the first row, but also division-by-zero if a previous value was exactly 0. Always use `NULLIF(LAG(amount) OVER (...), 0)` in the denominator of rate/change calculations.

**5. LAG offset assumes rows are contiguous — gaps in data cause wrong comparisons**
`LAG(monthly_total, 1)` assumes the previous row is the prior month. If January and March exist but February is missing, `LAG` of March compares against January without any warning. For sparse time series, join to a date spine and verify the lag date equals the expected prior period.

**6. Using a self join instead of LAG is much slower**
The self-join equivalent of LAG requires matching every row to its predecessor — O(n log n) or worse. `LAG` is computed in a single ordered pass — O(n). For any sequential comparison, prefer LAG/LEAD over self joins.


---
*sql_methods_library - Samantha McGarrigle*